# Kimi-Linear (GDN-2) Code LM — train on **Google Colab (T4)**

Trains the hybrid Gated-DeltaNet-2 / MLA / MoE code language model on **CodeParrot**
using the `configs/colab_t4.yaml` recipe (~190M params, bfloat16, single GPU).

**Before you start:** `Runtime -> Change runtime type -> Hardware accelerator = T4 GPU`.

The pipeline: install -> get code -> prepare data (tokenize) -> train -> evaluate -> generate.


## 0. Confirm the GPU

In [1]:
!nvidia-smi

Tue Jul  7 10:35:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Not running on Google Colab, skipping drive mount.")

Mounted at /content/drive


In [3]:
try:
    from huggingface_hub import notebook_login
    notebook_login()
except:
    print("Cannot login to Hugging Face Hub.")

## 1. Get the project

Set `REPO_URL` to your repository, or upload the project folder to `/content` and
skip the clone. Everything below assumes we `cd` into the project root.


In [1]:
import os

REPO_URL = "https://github.com/wisnunugroho21/nugie-codeparrot.git"  # <-- EDIT ME
PROJECT_DIR = "/content/nugie-codeparrot"

if not os.path.isdir(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR
%cd $PROJECT_DIR
!ls


/content/nugie-codeparrot
configs		  kimi_linear_gdn2.py	  pipeline     requirements.txt
data		  multi_latent_attention  __pycache__
gated_deltanet_2  notebooks		  README.md


In [2]:
%cd /content/nugie-codeparrot
!git pull

/content/nugie-codeparrot
Already up to date.


## 2. Install dependencies

We install the project's requirements first, then overlay the **CUDA 12 build of JAX**
last so the GPU plugin wins.


In [6]:
!pip install -q -U -r requirements.txt
!pip install -q -U "jax[cuda12]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.1/525.1 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

## 3. (Optional) Hugging Face token

CodeParrot streams fine anonymously, but a token raises the download rate limit. Skip
this cell if you don't have one.


In [7]:
# from huggingface_hub import login
# login()  # paste your HF token when prompted


## 4. Verify JAX sees the T4

In [3]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), "No GPU visible — set the runtime to T4."


JAX 0.10.2 devices: [CudaDevice(id=0)]


## 5. Build the config

We load `configs/colab_t4.yaml` and apply **demo-friendly overrides** (small corpus,
few steps) so this notebook finishes quickly. Delete the overrides for a real run.
We reuse this one `cfg` object for every stage below.


In [12]:
from pipeline.config import ExperimentConfig

cfg = ExperimentConfig.load("configs/colab_t4.yaml")

# --- Demo overrides (comment out for a full training run) ---
cfg.data.num_train_docs = 5000      # full config: 50000
cfg.data.num_val_docs   = 200
cfg.train.max_steps     = 5000      # full config: 40000
cfg.train.warmup_steps  = 100
cfg.train.eval_every    = 500
cfg.train.save_every    = 50
cfg.train.log_every     = 25
cfg.train.batch_size    = 4

# --- Optional: persist checkpoints to Google Drive (Colab runtimes are ephemeral) ---
from google.colab import drive; drive.mount("/content/drive")
cfg.train.ckpt_dir = "/content/drive/MyDrive/nugie-codeparrot/checkpoints/colab_t4"

cfg.validate()
print("compute_dtype:", cfg.model.compute_dtype, "| seq_len:", cfg.data.seq_len,
      "| batch:", cfg.train.batch_size, "| max_steps:", cfg.train.max_steps)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
compute_dtype: bfloat16 | seq_len: 512 | batch: 4 | max_steps: 5000


## 6. Prepare the data

Streams `codeparrot/codeparrot-clean` from the Hub, tokenizes with the CodeParrot BPE
tokenizer, and writes packed `train.bin` / `val.bin` memmaps + `meta.json`. Runs once.


In [ ]:
from pipeline import prepare_data
prepare_data.prepare(cfg)


[huggingface] codeparrot/codeparrot-clean tok=codeparrot vocab=32768 dtype=uint16


Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Tokenizing validation split...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2605 > 1024). Running this sequence through the model will result in indexing errors


Tokenizing training split...
  tokenized 1000 docs...
  tokenized 2000 docs...
  tokenized 3000 docs...
  tokenized 4000 docs...


## 7. Train

Streams metrics (loss / perplexity / aux / lr / tokens-per-sec) and writes Orbax
checkpoints to `cfg.train.ckpt_dir`. Re-run with `train.train(cfg, resume=True)` to
continue from the latest checkpoint.


In [ ]:
from pipeline import train
# train.train(cfg)
train.train(cfg, resume=True)

## 8. Evaluate

Restores the latest checkpoint and reports validation cross-entropy / perplexity /
bits-per-token over a capped number of batches.


In [ ]:
from pipeline import evaluate
evaluate.run_eval(cfg, step=None, max_batches=50)


## 9. Generate code

Autoregressive completion via the model's streaming decode. NOTE: after only the short
demo run above the output will be weak — train for many more steps for good code.


In [ ]:
evaluate.run_generate(cfg, step=None, prompt="def fibonacci(n):\n", max_new_tokens=128)
